# Main Figure Revision Sandbox

This notebook is a separate workspace for the requested figure edits without changing `make_main_figures_outline_v2.ipynb`.

Revision targets captured here:

- Figure 2: build either a 9-panel composite from PCA, boxplot, and line-plot views, or a cleaner 6-panel version if the line plot is redundant with the boxplots.
- Figure 4: replace cognitively heavy correlation-preservation panels with all-dataset summary metrics and focused feature-loss views.
- Figure 5: keep the existing noise plot direction.
- Figure 6: combine HIV, breast cancer, and diabetes reverse-ablation curves into one figure.
- Figure 1 schematic: source appears to be Matplotlib code, not a PowerPoint/Keynote file or GPT image. See the final schematic notes cell for editable options.


## Load Shared Definitions

This cell imports definitions from the outline notebook without running the expensive plotting/compute calls at the bottom of those cells. It also forces `RUN_MODE = "preview"` by default for quick iteration. Switch `REVISION_RUN_MODE` to `"final"` when you want manuscript-quality repeated runs.


In [ ]:
from pathlib import Path
import json
import pickle
import re

REVISION_RUN_MODE = "final"  # change to "final" for manuscript-quality compute
OUTLINE_NOTEBOOK = Path("data_synthesis/notebooks/make_main_figures_outline_v2.ipynb")


def _find_repo_root():
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for candidate in candidates:
        if (candidate / OUTLINE_NOTEBOOK).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {OUTLINE_NOTEBOOK} from {here}. "
        "Run this notebook from the repository or update OUTLINE_NOTEBOOK."
    )


repo_root = _find_repo_root()
outline_path = repo_root / OUTLINE_NOTEBOOK
CACHE_DIR = repo_root / "data_synthesis" / "notebooks" / "revision_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _trim_source(src, stop_patterns):
    lines = src.splitlines()
    kept = []
    for line in lines:
        if any(re.search(pattern, line) for pattern in stop_patterns):
            break
        kept.append(line)
    return "\n".join(kept) + "\n"


def _pick_code_cell(code_cells, startswith):
    for src in code_cells:
        if src.lstrip().startswith(startswith):
            return src
    raise KeyError(f"Could not find outline code cell starting with: {startswith}")


def load_outline_definitions(run_mode=REVISION_RUN_MODE):
    nb = json.loads(outline_path.read_text(encoding="utf-8"))
    code_cells = ["".join(cell.get("source", [])) for cell in nb["cells"] if cell.get("cell_type") == "code"]

    selected = []
    setup = _pick_code_cell(code_cells, "from pathlib import Path")
    selected.append(setup.replace('RUN_MODE = "final"', f'RUN_MODE = "{run_mode}"'))
    selected.append(_pick_code_cell(code_cells, "def _to_numpy_X"))
    selected.append(_trim_source(_pick_code_cell(code_cells, "def plot_figure2_cvae_pca"), [r"^plot_figure2_cvae_pca\("]))
    selected.append(_trim_source(_pick_code_cell(code_cells, "def one_run_origin_auc"), [r"^auc_runs\s*="]))
    selected.append(_trim_source(_pick_code_cell(code_cells, "def mean_kld_by_feature"), [r"^metric_table\s*="]))
    selected.append(_trim_source(_pick_code_cell(code_cells, "def feature_group"), [r"^plot_figure4_structure_panel\("]))
    selected.append(_trim_source(_pick_code_cell(code_cells, "def rank_discriminating_features"), [r"^ablation_df\s*="]))
    selected.append(_trim_source(_pick_code_cell(code_cells, "def plot_figure6_ablation_ac"), [r"^if \"ablation_df\" in globals\(\):"]))

    g = globals()
    for src in selected:
        exec(compile(src, str(outline_path), "exec"), g)

    # The imported setup cell defines its own repo_root based on the kernel cwd.
    # Restore the robust root found above so later export/cache paths are stable.
    g["repo_root"] = repo_root


load_outline_definitions()
print(f"Loaded outline definitions from {outline_path}")
print(f"Revision run mode: {RUN_MODE}; CVAE_EPOCHS={CVAE_EPOCHS}; AUC_REPEATS={AUC_REPEATS}; ABLATION_REPEATS={ABLATION_REPEATS}")
print(summary)


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\tonyt\\Desktop\\distinguishable_data\\data_synthesis\\notebooks\\data_synthesis\\notebooks\\make_main_figures_outline_v2.ipynb'

## Cached Compute Helpers

These wrappers avoid recomputing repeated RF probes, correlation summaries, and reverse ablation every time the notebook is opened. Set `force=True` in any helper call to regenerate.


In [ ]:
def _cache_path(name):
    return CACHE_DIR / f"{name}_{RUN_MODE}.pkl"


def _read_cache(name):
    path = _cache_path(name)
    if path.exists():
        with path.open("rb") as f:
            return pickle.load(f)
    return None


def _write_cache(name, obj):
    path = _cache_path(name)
    with path.open("wb") as f:
        pickle.dump(obj, f)
    return obj


def get_auc_runs(force=False):
    cached = None if force else _read_cache("auc_runs")
    if cached is not None:
        return cached
    return _write_cache("auc_runs", compute_auc_run_table(datasets))


def get_metric_table(force=False):
    cached = None if force else _read_cache("metric_table")
    if cached is not None:
        return cached
    auc_runs = get_auc_runs(force=force)
    return _write_cache("metric_table", build_metric_table(datasets, auc_runs))


def get_reverse_ablation(force=False):
    cached = None if force else _read_cache("ablation_df")
    if cached is not None:
        return cached
    return _write_cache("ablation_df", compute_reverse_ablation(datasets))


## Figure 2 Decision Note

If panel A of `figure3_linePlot_version.png` is the RF separability/AUC summary across datasets and methods, then yes: it is contained within panels A-C of `figure3_boxplot_version.png`. The boxplots show the repeated-run distributions, and the line plot is just a lower-information summary of those same values.

So the cleaner default here is a 6-panel Figure 2:

- A-C: CVAE PCA overlays for the three datasets.
- D-F: repeated RF separability distributions by method for the same three datasets.

A 9-panel option is still included below in case the line-plot row tells a useful narrative visually.


In [ ]:
def _plot_pca_panel(ax, ds, panel, seed=SEED, cvae_epochs=CVAE_EPOCHS):
    data = datasets[ds]
    X_real = np.asarray(data["X"], dtype=np.float32)
    X_syn, _ = sample_synthetic(ds, data, "CVAE", seed=seed, cvae_epochs=cvae_epochs)
    Xr, Xs = standardize_pair(X_real, X_syn)
    pca = PCA(n_components=2, random_state=seed).fit(Xr)
    Zr = pca.transform(Xr)
    Zs = pca.transform(Xs)
    rng = np.random.default_rng(seed)
    Zs_plot = Zs[rng.choice(len(Zs), size=700, replace=False)] if len(Zs) > 700 else Zs

    ax.scatter(Zr[:, 0], Zr[:, 1], s=8, marker="o", facecolors="none", alpha=0.58,
               edgecolors="#8A8A8A", linewidths=0.55, label="Real")
    ax.scatter(Zs_plot[:, 0], Zs_plot[:, 1], s=8, marker="o", color=DATASET_COLORS[ds], alpha=0.74,
               edgecolors="none", label="CVAE")
    add_confidence_ellipse(ax, Zr, "#8A8A8A", linewidth=1.8)
    add_confidence_ellipse(ax, Zs, DATASET_COLORS[ds], linewidth=2.0)

    all_z = np.vstack([Zr, Zs_plot])
    x_min, x_max = np.nanmin(all_z[:, 0]), np.nanmax(all_z[:, 0])
    y_min, y_max = np.nanmin(all_z[:, 1]), np.nanmax(all_z[:, 1])
    ax.set_xlim(x_min - (x_max - x_min) * 0.24, x_max + (x_max - x_min) * 0.24)
    ax.set_ylim(y_min - (y_max - y_min) * 0.24, y_max + (y_max - y_min) * 0.24)

    ev = pca.explained_variance_ratio_
    ax.set_title(ds, color=DATASET_COLORS[ds], weight="semibold", pad=8)
    ax.set_xlabel(f"PC1 ({ev[0] * 100:.1f}%)")
    ax.set_ylabel(f"PC2 ({ev[1] * 100:.1f}%)")
    ax.text(0.045, 0.055, f"n={len(data['y'])}, p={X_real.shape[1]}", transform=ax.transAxes,
            color=DATASET_COLORS[ds], fontsize=8.5, weight="bold", ha="left", va="bottom")
    ax.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#9A9A9A", fontsize=9.0,
              borderpad=0.70, labelspacing=0.45, handlelength=1.7, handletextpad=0.60,
              markerscale=1.25, framealpha=0.96)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.2)
    ax.tick_params(labelsize=8.5, width=1.2, length=4)
    add_panel_label(ax, panel)


def _plot_auc_box_panel(ax, auc_runs, ds, panel):
    sub = auc_runs[auc_runs["dataset"] == ds]
    values = [sub[sub["method"] == method]["separability_auc"].to_numpy() for method in METHOD_ORDER]
    bp = ax.boxplot(values, patch_artist=True, showmeans=True, widths=0.62, tick_labels=METHOD_ORDER)
    for patch, method in zip(bp["boxes"], METHOD_ORDER):
        patch.set_facecolor(METHOD_PASTELS[method])
        patch.set_edgecolor(METHOD_COLORS[method])
        patch.set_linewidth(1.7)
    for key in ["whiskers", "caps", "medians", "means"]:
        for artist in bp.get(key, []):
            artist.set_color(NEUTRAL)
            artist.set_linewidth(1.35)
    ax.axhline(0.5, color="#777777", linestyle="--", linewidth=1.2)
    ax.set_title(f"{ds}: RF separability", color=DATASET_COLORS[ds], weight="bold", pad=8)
    ax.set_ylabel("AUC")
    ax.tick_params(axis="x", rotation=25, labelsize=8.5)
    clean_axis(ax, grid_axis="y")
    add_panel_label(ax, panel)


def _plot_auc_line_panel(ax, auc_runs, ds, panel):
    sub = auc_runs[auc_runs["dataset"] == ds]
    means = sub.groupby("method")["separability_auc"].mean().reindex(METHOD_ORDER)
    sds = sub.groupby("method")["separability_auc"].std().reindex(METHOD_ORDER)
    x = np.arange(len(METHOD_ORDER))
    for i, method in enumerate(METHOD_ORDER):
        ax.errorbar(i, means.loc[method], yerr=sds.loc[method], marker="o", markersize=7,
                    color=METHOD_COLORS[method], capsize=4, linewidth=2.0)
    ax.plot(x, means.to_numpy(), color=DATASET_COLORS[ds], linewidth=2.0, alpha=0.65)
    ax.axhline(0.5, color="#777777", linestyle="--", linewidth=1.2)
    ax.set_xticks(x)
    ax.set_xticklabels(METHOD_ORDER, rotation=25, ha="right", fontsize=8.5)
    ax.set_title(f"{ds}: mean +/- SD", color=DATASET_COLORS[ds], weight="bold", pad=8)
    ax.set_ylabel("AUC")
    clean_axis(ax, grid_axis="y")
    add_panel_label(ax, panel)


def plot_figure2_six_panel(auc_runs):
    fig, axes = plt.subplots(2, 3, figsize=(13.8, 8.0))
    for ax, ds, panel in zip(axes[0], DATASET_ORDER, ["A", "B", "C"]):
        _plot_pca_panel(ax, ds, panel)
    for ax, ds, panel in zip(axes[1], DATASET_ORDER, ["D", "E", "F"]):
        _plot_auc_box_panel(ax, auc_runs, ds, panel)
    fig.suptitle("Figure 2. Synthetic data geometry and separability", y=0.99, fontsize=15, weight="semibold")
    fig.subplots_adjust(left=0.065, right=0.99, top=0.92, bottom=0.10, wspace=0.30, hspace=0.42)
    return fig


def plot_figure2_nine_panel(auc_runs):
    fig, axes = plt.subplots(3, 3, figsize=(13.8, 11.4))
    panels = list("ABCDEFGHI")
    for ax, ds, panel in zip(axes[0], DATASET_ORDER, panels[:3]):
        _plot_pca_panel(ax, ds, panel)
    for ax, ds, panel in zip(axes[1], DATASET_ORDER, panels[3:6]):
        _plot_auc_box_panel(ax, auc_runs, ds, panel)
    for ax, ds, panel in zip(axes[2], DATASET_ORDER, panels[6:]):
        _plot_auc_line_panel(ax, auc_runs, ds, panel)
    fig.suptitle("Figure 2. PCA geometry, separability distributions, and summary trends", y=0.99, fontsize=15, weight="semibold")
    fig.subplots_adjust(left=0.065, right=0.99, top=0.94, bottom=0.08, wspace=0.30, hspace=0.50)
    return fig


In [ ]:
auc_runs = get_auc_runs(force=False)
fig2_6 = plot_figure2_six_panel(auc_runs)
# fig2_9 = plot_figure2_nine_panel(auc_runs)


## Figure 4: Correlation-Structure Preservation

Goal: summarize whether synthetic data preserves feature interdependency structure without showing overwhelming full correlation matrices.

Plain-language readout:

- **Structure retained**: compares all feature-feature correlations in the real data with the same feature-feature correlations in the synthetic data. Higher means the synthetic dataset keeps the same correlation pattern as the real dataset.
- **Structure changed**: averages how much those feature-feature correlations moved. Lower means less distortion.
- **Pairs preserved within 0.10**: the fraction of feature pairs whose correlation changed by no more than 0.10. Higher means more of the correlation structure survived.

Math context for the caption: for each dataset and synthetic method, compute the real feature correlation matrix and synthetic feature correlation matrix. Then compare only the off-diagonal feature pairs. If a real pair has correlation 0.70 and synthetic has 0.45, that pair changed by `|0.70 - 0.45| = 0.25`.


In [ ]:
def feature_corr_loss_scores(real_corr, syn_corr, feature_names):
    delta = np.abs(real_corr - syn_corr).astype(float)
    np.fill_diagonal(delta, np.nan)
    scores = np.nanmean(delta, axis=1)
    return pd.DataFrame({"feature": feature_names, "structure_change_score": scores}).sort_values(
        "structure_change_score", ascending=False
    )


def top_changed_correlation_pairs(real_corr, syn_corr, feature_names, top_n=8):
    rows = []
    for i in range(len(feature_names)):
        for j in range(i + 1, len(feature_names)):
            real_val = float(real_corr[i, j])
            syn_val = float(syn_corr[i, j])
            rows.append({
                "feature_1": feature_names[i],
                "feature_2": feature_names[j],
                "real_corr": real_val,
                "synthetic_corr": syn_val,
                "abs_change": abs(real_val - syn_val),
            })
    return pd.DataFrame(rows).sort_values("abs_change", ascending=False).head(top_n)


def compute_correlation_preservation(datasets, seed=SEED, cvae_epochs=CVAE_EPOCHS, retain_tol=0.10):
    rows = []
    feature_rows = []
    pair_rows = []
    for ds in DATASET_ORDER:
        data = datasets[ds]
        X_real = np.asarray(data["X"], dtype=np.float32)
        feature_names = list(data.get("feature_names", [f"f{i}" for i in range(X_real.shape[1])]))
        real_corr = corr_matrix(X_real)
        real_upper = upper_triangle_values(real_corr)
        for method in METHOD_ORDER:
            print(f"[correlation] {ds} - {method}")
            X_syn, _ = sample_synthetic(ds, data, method, seed=seed, cvae_epochs=cvae_epochs)
            syn_corr = corr_matrix(X_syn)
            syn_upper = upper_triangle_values(syn_corr)
            delta = np.abs(real_upper - syn_upper)
            corr_pair_r = float(np.corrcoef(real_upper, syn_upper)[0, 1]) if np.std(real_upper) > 0 and np.std(syn_upper) > 0 else np.nan
            rows.append({
                "dataset": ds,
                "method": method,
                "structure_retained": corr_pair_r,
                "structure_change_score": float(np.mean(delta)),
                "median_abs_change": float(np.median(delta)),
                "p90_abs_change": float(np.percentile(delta, 90)),
                "pairs_preserved_0p10": float(np.mean(delta <= retain_tol)),
            })
            scores = feature_corr_loss_scores(real_corr, syn_corr, feature_names)
            scores.insert(0, "method", method)
            scores.insert(0, "dataset", ds)
            feature_rows.append(scores)

            pairs = top_changed_correlation_pairs(real_corr, syn_corr, feature_names, top_n=8)
            pairs.insert(0, "method", method)
            pairs.insert(0, "dataset", ds)
            pair_rows.append(pairs)
    return pd.DataFrame(rows), pd.concat(feature_rows, ignore_index=True), pd.concat(pair_rows, ignore_index=True)


def get_correlation_preservation(force=False):
    cached = None if force else _read_cache("corr_preservation_v2")
    if cached is not None:
        return cached
    result = compute_correlation_preservation(datasets)
    return _write_cache("corr_preservation_v2", result)


def plot_figure4_correlation_summary(corr_summary):
    fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.8), sharey=True)
    metrics = [
        ("structure_retained", "Structure retained", "higher is better", (0.0, 1.0)),
        ("structure_change_score", "Structure changed", "lower is better", None),
        ("pairs_preserved_0p10", "Pairs preserved", "higher is better", (0.0, 1.0)),
    ]
    x = np.arange(len(DATASET_ORDER))
    offsets = np.linspace(-0.27, 0.27, len(METHOD_ORDER))

    for ax, (metric, title, direction, ylim), panel in zip(axes, metrics, ["A", "B", "C"]):
        for offset, method in zip(offsets, METHOD_ORDER):
            sub = corr_summary[corr_summary["method"] == method].set_index("dataset").reindex(DATASET_ORDER)
            ax.scatter(x + offset, sub[metric], s=78, color=METHOD_COLORS[method], edgecolor="white",
                       linewidth=0.9, label=method, zorder=3)
            ax.plot(x + offset, sub[metric], color=METHOD_COLORS[method], linewidth=1.2, alpha=0.45, zorder=2)
        ax.set_title(title, weight="bold", pad=8)
        ax.text(0.5, -0.32, direction, transform=ax.transAxes, ha="center", va="top",
                fontsize=10.0, color="#555555")
        ax.set_xticks(x)
        ax.set_xticklabels(DATASET_ORDER, rotation=18, ha="right")
        if ylim is not None:
            ax.set_ylim(*ylim)
        clean_axis(ax, grid_axis="y")
        add_panel_label(ax, panel)

    axes[0].set_ylabel("Metric value")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.02), ncol=len(METHOD_ORDER),
               frameon=True, facecolor="white", edgecolor="black", framealpha=1.0)
    fig.suptitle("Figure 4. Synthetic data alters feature-correlation structure", y=0.98, fontsize=16, weight="bold")
    fig.text(
        0.5, 0.005,
        "Each point summarizes all off-diagonal feature-pair correlations for one dataset and generator. "
        "Structure changed is the average absolute movement in correlation values; smaller values indicate less distortion.",
        ha="center", va="bottom", fontsize=9.5, color="#444444",
    )
    fig.subplots_adjust(left=0.075, right=0.99, top=0.78, bottom=0.30, wspace=0.25)
    return fig


def make_top_changed_pair_table(changed_pairs, top_n=5):
    out = (
        changed_pairs
        .sort_values(["dataset", "method", "abs_change"], ascending=[True, True, False])
        .groupby(["dataset", "method"])
        .head(top_n)
        .copy()
    )
    for col in ["real_corr", "synthetic_corr", "abs_change"]:
        out[col] = out[col].round(3)
    return out


In [ ]:
corr_summary, feature_corr_loss, changed_corr_pairs = get_correlation_preservation(force=False)
display(corr_summary.sort_values(["dataset", "method"]))
fig4_summary = plot_figure4_correlation_summary(corr_summary)

# Diagnostic table, not intended as a main figure panel: which feature pairs changed most?
top_changed_pair_table = make_top_changed_pair_table(changed_corr_pairs, top_n=5)
display(top_changed_pair_table)


## Figure 5: Noise Sensitivity

Repeated here from the outline notebook so this revision notebook can generate the full figure set in one place. Synthetic data are perturbed by increasing Gaussian noise, and RF real-vs-synthetic separability is recomputed across datasets and generators.


In [ ]:
def stratified_subsample(X, y, n0, n1, seed=SEED):
    rng = np.random.default_rng(seed)
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    take0 = rng.choice(idx0, size=max(1, min(n0, len(idx0))), replace=False)
    take1 = rng.choice(idx1, size=max(1, min(n1, len(idx1))), replace=False)
    idx = np.r_[take0, take1]
    rng.shuffle(idx)
    return X[idx], y[idx]


def compute_noise_sensitivity(datasets, seed=SEED, repeats=NOISE_REPEATS, sigmas=PROBE_SIGMAS, frac=NOISE_FRAC, cvae_epochs=CVAE_EPOCHS):
    rows = []
    for ds in DATASET_ORDER:
        data = datasets[ds]
        X = np.asarray(data["X"], dtype=np.float32)
        y = np.asarray(data["y"], dtype=int)
        n0 = max(2, int((y == 0).sum() * frac))
        n1 = max(2, int((y == 1).sum() * frac))
        X_sub, y_sub = stratified_subsample(X, y, n0, n1, seed=seed)
        stds = X_sub.std(axis=0)
        stds = np.where(stds == 0, 1.0, stds)
        feat_cols = [f"f{i}" for i in range(X_sub.shape[1])]
        print(f"[noise] training CVAE for {ds}")
        state = train_cvae(X_sub, y_sub, cfg=Config(seed=seed, epochs=cvae_epochs, batch_size=32), verbose=False)
        generators = {
            "Bootstrap": lambda s: sample_bootstrap(X_sub, y_sub, n0, n1, seed=s),
            "Column-wise": lambda s: sample_columnwise(X_sub, y_sub, n0, n1, seed=s),
            "GMM": lambda s: sample_gmm(X_sub, y_sub, n0, n1, seed=s),
            "CVAE": lambda s: sample_trained_cvae(state, n0, n1, seed=s),
        }
        for method, gen in generators.items():
            for sigma in sigmas:
                vals = []
                for r in range(repeats):
                    rep_seed = seed + r
                    X_syn, _ = gen(rep_seed)
                    X_syn = np.asarray(X_syn, dtype=np.float64)
                    if sigma > 0:
                        rng = np.random.default_rng(rep_seed + 500)
                        X_syn = X_syn + rng.standard_normal(X_syn.shape) * stds * sigma
                    real_df = pd.DataFrame(X_sub, columns=feat_cols)
                    real_df["target"] = 1
                    syn_df = pd.DataFrame(X_syn, columns=feat_cols)
                    syn_df["target"] = 0
                    combined = pd.concat([real_df, syn_df], ignore_index=True)
                    avg, _, _ = RFWrapper.from_combined(combined)
                    vals.append(max(float(avg), 1.0 - float(avg)))
                rows.append({"dataset": ds, "method": method, "sigma": sigma,
                             "sep_mean": float(np.mean(vals)), "sep_sd": float(np.std(vals)),
                             "sep_values": [float(v) for v in vals]})
                print(f"[noise] {ds} {method} sigma={sigma}")
    return pd.DataFrame(rows)


def get_noise_sensitivity(force=False):
    cached = None if force else _read_cache("noise_df")
    if cached is not None:
        return cached
    result = compute_noise_sensitivity(datasets)
    return _write_cache("noise_df", result)


def plot_figure5_noise(noise_df):
    fig, axes = plt.subplots(1, 3, figsize=(14.8, 4.9), sharey=True, constrained_layout=False)
    legend_handles = []
    for ax, ds, panel in zip(axes, DATASET_ORDER, ["A", "B", "C"]):
        sub = noise_df[noise_df["dataset"] == ds]
        for method in METHOD_ORDER:
            m = sub[sub["method"] == method].sort_values("sigma")
            if m.empty:
                continue
            line, = ax.plot(m["sigma"], m["sep_mean"], marker="o", color=METHOD_COLORS[method], label=method,
                            linewidth=2.5, markersize=5.8)
            ax.fill_between(m["sigma"], m["sep_mean"] - m["sep_sd"], m["sep_mean"] + m["sep_sd"],
                            color=METHOD_COLORS[method], alpha=0.12, linewidth=0)
            if ax is axes[0]:
                legend_handles.append(line)
        ax.axhline(0.5, color="#777777", linestyle="--", linewidth=1.35)
        ax.set_title(ds, color=DATASET_COLORS[ds], weight="semibold", fontsize=13)
        ax.set_xlabel("Noise (sigma)")
        clean_axis(ax, grid_axis="y")
        ax.spines["left"].set_linewidth(1.8)
        ax.spines["bottom"].set_linewidth(1.8)
        ax.tick_params(width=1.4, length=5)
        add_panel_label(ax, panel)
    axes[0].set_ylabel("AUC")
    fig.legend(legend_handles, METHOD_ORDER, loc="lower center", bbox_to_anchor=(0.5, 0.01), ncol=len(METHOD_ORDER),
               frameon=True, facecolor="white", edgecolor="black", framealpha=1.0, borderpad=0.55)
    fig.suptitle("Noise sensitivity", y=0.98, fontsize=15, weight="semibold")
    fig.subplots_adjust(left=0.07, right=0.99, top=0.82, bottom=0.22, wspace=0.25)
    return fig


noise_df = get_noise_sensitivity(force=False)
fig5_noise = plot_figure5_noise(noise_df)


## Figure 6: Single Combined Reverse-Ablation Figure

This uses the all-dataset A-C function added to the outline notebook, and keeps the x-axis as percentage of top discriminator-ranked features removed so datasets with different numbers of features are comparable.


In [ ]:
def _ablation_progress_percent(m):
    removable = (m["n_features_removed"] + m["n_features_retained"] - 2).clip(lower=1)
    return 100 * m["n_features_removed"] / removable


def _flat_values(series):
    vals = []
    for item in series:
        vals.extend(item if isinstance(item, (list, tuple, np.ndarray)) else [item])
    return np.asarray(vals, dtype=float)


def plot_figure6_ablation_all_datasets(ablation_df):
    fig = plt.figure(figsize=(13.4, 7.0), constrained_layout=False)
    gs = fig.add_gridspec(2, len(DATASET_ORDER), height_ratios=[1.35, 0.95])
    curve_axes = [fig.add_subplot(gs[0, i]) for i in range(len(DATASET_ORDER))]
    box_axes = [fig.add_subplot(gs[1, i], sharey=curve_axes[0]) for i in range(len(DATASET_ORDER))]

    legend_handles = []
    for panel_idx, (ax_curve, ax_box, ds) in enumerate(zip(curve_axes, box_axes, DATASET_ORDER)):
        sub = ablation_df[ablation_df["dataset"] == ds]
        box_values, box_labels = [], []
        if sub.empty:
            ax_curve.set_visible(False)
            ax_box.set_visible(False)
            continue

        for method in METHOD_ORDER:
            m = sub[sub["method"] == method].sort_values("n_features_removed")
            if m.empty:
                continue
            pct_removed = _ablation_progress_percent(m)
            line, = ax_curve.plot(pct_removed, m["auc_mean"], color=METHOD_COLORS[method], marker="o",
                                  linewidth=2.35, markersize=5.2, label=method)
            ax_curve.fill_between(pct_removed, m["auc_mean"] - m["auc_sd"], m["auc_mean"] + m["auc_sd"],
                                  color=METHOD_COLORS[method], alpha=0.12, linewidth=0)
            if panel_idx == 0:
                legend_handles.append(line)
            box_values.append(_flat_values(m["auc_values"]))
            box_labels.append(method)

        ax_curve.axhline(0.5, color="#777777", linestyle="--", linewidth=1.25)
        ax_curve.set_title(ds, color=DATASET_COLORS[ds], weight="semibold", pad=8, fontsize=13)
        ax_curve.text(-0.13, 1.08, chr(ord("A") + panel_idx), transform=ax_curve.transAxes,
                      fontsize=15, weight="bold", va="top", ha="left")
        ax_curve.set_xlim(-2, 102)
        ax_curve.set_xticks([0, 50, 100])
        ax_curve.set_xlabel("Removed (%)")
        clean_axis(ax_curve, grid_axis="y")
        ax_curve.spines["left"].set_linewidth(1.8)
        ax_curve.spines["bottom"].set_linewidth(1.8)
        ax_curve.tick_params(width=1.4, length=5)

        bp = ax_box.boxplot(box_values, tick_labels=box_labels, patch_artist=True, showmeans=True, widths=0.60)
        for patch, method in zip(bp["boxes"], box_labels):
            patch.set_facecolor(METHOD_PASTELS[method])
            patch.set_edgecolor(METHOD_COLORS[method])
            patch.set_linewidth(1.55)
        for key in ["whiskers", "caps", "medians", "means"]:
            for artist in bp.get(key, []):
                artist.set_color(NEUTRAL)
                artist.set_linewidth(1.25)
        ax_box.axhline(0.5, color="#777777", linestyle="--", linewidth=1.25)
        ax_box.tick_params(axis="x", rotation=25, labelsize=9.0, width=1.4, length=5)
        clean_axis(ax_box, grid_axis="y")
        ax_box.spines["left"].set_linewidth(1.8)
        ax_box.spines["bottom"].set_linewidth(1.8)

    curve_axes[0].set_ylabel("AUC")
    box_axes[0].set_ylabel("AUC")
    for ax in curve_axes[1:] + box_axes[1:]:
        ax.tick_params(labelleft=False)

    y_values = []
    for _, row in ablation_df.iterrows():
        y_values.extend([row["auc_mean"] - row["auc_sd"], row["auc_mean"] + row["auc_sd"]])
        y_values.extend(row.get("auc_values", []))
    y_min = max(0.45, np.nanmin(y_values) - 0.03)
    y_max = min(1.02, np.nanmax(y_values) + 0.03)
    for ax in curve_axes + box_axes:
        if ax.get_visible():
            ax.set_ylim(y_min, y_max)

    handles = legend_handles[:len(METHOD_ORDER)]
    fig.legend(handles, [h.get_label() for h in handles], loc="lower center",
               bbox_to_anchor=(0.5, 0.01), ncol=len(handles), frameon=True,
               facecolor="white", edgecolor="black", framealpha=1.0, borderpad=0.55)
    fig.suptitle("Reverse feature ablation", y=0.98, fontsize=15, weight="semibold")
    fig.text(0.5, 0.075, "Removed (%) is scaled to the maximum tested ablation while retaining two features.",
             ha="center", fontsize=9.0, color="#444444")
    fig.subplots_adjust(left=0.075, right=0.99, top=0.88, bottom=0.20, wspace=0.18, hspace=0.35)
    return fig


ablation_df = get_reverse_ablation(force=False)
fig6_ac = plot_figure6_ablation_all_datasets(ablation_df)


## Export Helpers

Exports are intentionally explicit so we do not overwrite manuscript files by accident.


In [ ]:
EXPORT_DIR = repo_root / "data_synthesis" / "notebooks" / "revision_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Uncomment after running the corresponding figure cells.
# fig2_6.savefig(EXPORT_DIR / "figure2_six_panel_revision.png", dpi=300, bbox_inches="tight")
# fig2_9.savefig(EXPORT_DIR / "figure2_nine_panel_revision.png", dpi=300, bbox_inches="tight")
# fig4_summary.savefig(EXPORT_DIR / "figure4_correlation_summary_revision.png", dpi=300, bbox_inches="tight")
# fig5_noise.savefig(EXPORT_DIR / "figure5_noise_revision.png", dpi=300, bbox_inches="tight")
# fig6_ac.savefig(EXPORT_DIR / "figure6_ablation_all_datasets_revision.png", dpi=300, bbox_inches="tight")

SCHEMATIC_PPTX = EXPORT_DIR / "schematic_workflow_editable.pptx"
SCHEMATIC_PPTX


## Schematic Edit Note

The schematic in `make_main_figures_outline_v2.ipynb` appears to be generated with Matplotlib patches and text, not GPT image generation and not PowerPoint/Keynote. I created an editable PowerPoint version using native PPT shapes at:

`data_synthesis/notebooks/revision_exports/schematic_workflow_editable.pptx`

All section boxes, text, arrows, markers, and mini-panels are editable PowerPoint objects. `python-pptx` is included in the project dependencies, so we can keep iterating the schematic as a PPT source if that is easier for font-size and flow tuning.
